# 01 Data Understanding

This notebook performs a clean, reproducible EDA pass for the AI Immune System Challenge dataset.

Scope:
- Data loading and schema checks
- JSONL structure understanding
- Label analysis
- Text/conversation analysis
- Qualitative inspection
- Train/test/submission format comparison
- Final EDA findings

Non-goals for this notebook:
- No model training
- No submission generation


In [2]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

SEED = 42
DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


In [3]:
train_path = DATA_DIR / "train_labeled_comp.jsonl"
test_path = DATA_DIR / "test_labeled_comp.jsonl"
solution_path = DATA_DIR / "solution_format.csv"

train_df = pd.read_json(train_path, lines=True)
test_df = pd.read_json(test_path, lines=True)
solution_df = pd.read_csv(solution_path)

print("Loaded files:")
print(f"- {train_path}")
print(f"- {test_path}")
print(f"- {solution_path}")


Loaded files:
- ../data/train_labeled_comp.jsonl
- ../data/test_labeled_comp.jsonl
- ../data/solution_format.csv


In [4]:
def basic_info(df: pd.DataFrame, name: str):
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print("dtypes:")
    print(df.dtypes)
    print("first 5 rows:")
    print(df.head())
    print("missing values:")
    print(df.isna().sum())
    print("duplicate rows:", int(df.duplicated().sum()))

    id_cols = [c for c in df.columns if c.lower() == "id" or c.lower().endswith("_id")]
    if id_cols:
        for col in id_cols:
            print(f"duplicate ids in {col}:", int(df[col].duplicated().sum()))
    else:
        print("duplicate ids: no id-like column found")


basic_info(train_df, "train")
basic_info(test_df, "test")
basic_info(solution_df, "solution_format")



=== train ===
shape: (4900, 2)
columns: ['label', 'text']
dtypes:
label    str
text     str
dtype: object
first 5 rows:
   label                                               text
0  FALSE  Although some of its components intersect deep...
1  FALSE  Losing excess WEIGHT can help jokers take off ...
2  FALSE  Crazy how many types dayalilyami has them! Bas...
3  FALSE  Inevitably someone will lose pollen, use the w...
4  FALSE  Topics might even be blurred through this proc...
missing values:
label    0
text     0
dtype: int64
duplicate rows: 0
duplicate ids: no id-like column found

=== test ===
shape: (2100, 1)
columns: ['text']
dtypes:
text    str
dtype: object
first 5 rows:
                                                text
0  Make adjustments monthly or quarterly as impro...
1  So whether exploring applications designed for...
2  Sleep deprivation further worsens the mental d...
3  Create me one please!" What person''s planner ...
4  Based this reading, how funable will the activ

In [5]:
with open(train_path, "r", encoding="utf-8") as f:
    raw_train_row = json.loads(f.readline())

with open(test_path, "r", encoding="utf-8") as f:
    raw_test_row = json.loads(f.readline())

print("Raw train row (first line):")
print(raw_train_row)
print("\nRaw test row (first line):")
print(raw_test_row)

def find_text_like_columns(columns):
    keywords = ("text", "conversation", "dialog", "message", "content", "prompt", "response")
    matches = []
    for c in columns:
        c_low = c.lower()
        if any(k in c_low for k in keywords):
            matches.append(c)
    return matches

train_text_like_cols = find_text_like_columns(train_df.columns)
test_text_like_cols = find_text_like_columns(test_df.columns)

print("\nText-like columns in train:", train_text_like_cols)
print("Text-like columns in test:", test_text_like_cols)

for col in train_text_like_cols:
    type_counts = train_df[col].head(200).map(lambda x: type(x).__name__).value_counts()
    print(f"\nTrain column `{col}` value types (sample of 200):")
    print(type_counts)

for col in test_text_like_cols:
    type_counts = test_df[col].head(200).map(lambda x: type(x).__name__).value_counts()
    print(f"\nTest column `{col}` value types (sample of 200):")
    print(type_counts)


Raw train row (first line):
{'label': 'FALSE', 'text': 'Although some of its components intersect deeper into its mechanics and science, the game of chess is still considered a form of art.'}

Raw test row (first line):
{'text': 'Make adjustments monthly or quarterly as improvementsPushMatrix are made to address accountability, communication, and decision-making.'}

Text-like columns in train: ['text']
Text-like columns in test: ['text']

Train column `text` value types (sample of 200):
text
str    200
Name: count, dtype: int64

Test column `text` value types (sample of 200):
text
str    200
Name: count, dtype: int64


In [6]:
def normalize_label(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"
    text = str(value).strip().upper()
    if text in {"TRUE", "T", "1"}:
        return "TRUE"
    if text in {"FALSE", "F", "0"}:
        return "FALSE"
    return text


def safe_extract_text(sample):
    # Safely flatten nested objects into readable text without mutating raw structure.
    if sample is None:
        return ""
    if isinstance(sample, str):
        return sample
    if isinstance(sample, (int, float, bool)):
        return str(sample)
    if isinstance(sample, list):
        parts = [safe_extract_text(item) for item in sample]
        parts = [p for p in parts if p]
        return "\n".join(parts)
    if isinstance(sample, dict):
        preferred_keys = ["text", "content", "message", "value", "prompt", "response"]
        parts = []
        for key in preferred_keys:
            if key in sample:
                extracted = safe_extract_text(sample.get(key))
                if extracted:
                    parts.append(extracted)
        if parts:
            return "\n".join(parts)
        fallback = [safe_extract_text(v) for v in sample.values()]
        fallback = [p for p in fallback if p]
        return "\n".join(fallback)
    return str(sample)


def approx_turn_count(raw_sample, text_value):
    if isinstance(raw_sample, list):
        return len(raw_sample)
    if isinstance(raw_sample, dict):
        for key in ["messages", "conversation", "turns", "dialogue"]:
            if key in raw_sample and isinstance(raw_sample[key], list):
                return len(raw_sample[key])
        return 1 if text_value else 0
    if not text_value:
        return 0
    role_markers = re.findall(r"(?im)^\s*(user|assistant|system)\s*:", text_value)
    if role_markers:
        return len(role_markers)
    non_empty_lines = [line for line in text_value.splitlines() if line.strip()]
    return max(1, len(non_empty_lines))


def url_count(text):
    if not text:
        return 0
    return len(re.findall(r"https?://\S+|www\.\S+", text, flags=re.IGNORECASE))


def code_like_count(text):
    if not text:
        return 0
    fenced_blocks = len(re.findall(r"```[\s\S]*?```", text))
    inline_code = len(re.findall(r"`[^`]+`", text))
    code_keywords = len(re.findall(r"\b(def|class|import|from|SELECT|INSERT|UPDATE|DELETE|DROP|curl|wget|sudo|#!/bin/bash)\b", text, flags=re.IGNORECASE))
    html_tags = len(re.findall(r"<\/?[a-zA-Z][^>]*>", text))
    return fenced_blocks + inline_code + code_keywords + html_tags


In [7]:
print("Train label value Python types:")
print(train_df["label"].map(lambda x: type(x).__name__).value_counts(dropna=False))

train_eda = train_df.copy()
train_eda["label_normalized"] = train_eda["label"].map(normalize_label)

label_distribution = (
    train_eda["label_normalized"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)
label_distribution["percentage"] = (label_distribution["count"] / len(train_eda) * 100).round(4)

print("\nLabel distribution:")
print(label_distribution)

label_distribution.to_csv(OUTPUT_DIR / "label_distribution.csv", index=False)
print("\nSaved:", OUTPUT_DIR / "label_distribution.csv")


Train label value Python types:
label
str    4900
Name: count, dtype: int64

Label distribution:
   label  count  percentage
0  FALSE   3500     71.4286
1   TRUE   1400     28.5714

Saved: ../outputs/label_distribution.csv


In [8]:
train_eda["derived_text"] = train_eda["text"].apply(safe_extract_text)
test_eda = test_df.copy()
test_eda["derived_text"] = test_eda["text"].apply(safe_extract_text)

for df in [train_eda, test_eda]:
    df["char_length"] = df["derived_text"].str.len().fillna(0).astype(int)
    df["word_count"] = df["derived_text"].str.findall(r"\b\w+\b").str.len().fillna(0).astype(int)
    df["approx_turn_count"] = [approx_turn_count(raw, txt) for raw, txt in zip(df["text"], df["derived_text"])]
    df["newline_count"] = df["derived_text"].str.count(r"\n").fillna(0).astype(int)
    df["url_count"] = df["derived_text"].apply(url_count).astype(int)
    df["code_like_count"] = df["derived_text"].apply(code_like_count).astype(int)

metric_cols = ["char_length", "word_count", "approx_turn_count", "newline_count", "url_count", "code_like_count"]

text_stats_by_label = (
    train_eda.groupby("label_normalized")[metric_cols]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(4)
)
text_stats_by_label.columns = [f"{metric}_{stat}" for metric, stat in text_stats_by_label.columns]
text_stats_by_label = text_stats_by_label.reset_index()

print("Text stats by label:")
print(text_stats_by_label)

text_stats_by_label.to_csv(OUTPUT_DIR / "text_stats_by_label.csv", index=False)
print("\nSaved:", OUTPUT_DIR / "text_stats_by_label.csv")

print("\nTrain vs Test length comparison (describe):")
length_compare = pd.DataFrame({
    "train_char_length": train_eda["char_length"].describe(),
    "test_char_length": test_eda["char_length"].describe(),
    "train_word_count": train_eda["word_count"].describe(),
    "test_word_count": test_eda["word_count"].describe(),
})
print(length_compare)


Text stats by label:
  label_normalized  char_length_count  char_length_mean  char_length_median  char_length_std  char_length_min  char_length_max  word_count_count  \
0            FALSE               3500          132.8811               120.0          72.1926               16             1163              3500   
1             TRUE               1400          188.0121               173.0          90.7854               17             1135              1400   

   word_count_mean  word_count_median  word_count_std  word_count_min  word_count_max  approx_turn_count_count  approx_turn_count_mean  \
0          22.4240               20.0         12.8586               2             212                     3500                  1.3586   
1          31.1779               28.0         17.1022               3             222                     1400                  1.5286   

   approx_turn_count_median  approx_turn_count_std  approx_turn_count_min  approx_turn_count_max  newline_count_count  

In [9]:
def print_sample_examples(df, label_value, n=5):
    subset = df[df["label_normalized"] == label_value]
    sample_df = subset.sample(n=min(n, len(subset)), random_state=SEED)
    print(f"\nRandom samples for label={label_value} (n={len(sample_df)}):")
    for idx, row in sample_df.iterrows():
        print("-" * 100)
        print(f"index: {idx}")
        print(f"char_length: {row['char_length']}, word_count: {row['word_count']}, approx_turn_count: {row['approx_turn_count']}")
        print(row["derived_text"][:800])


print_sample_examples(train_eda, "TRUE", n=5)
print_sample_examples(train_eda, "FALSE", n=5)

for label_value in ["TRUE", "FALSE"]:
    subset = train_eda[train_eda["label_normalized"] == label_value].copy()
    shortest = subset.nsmallest(1, "char_length")
    longest = subset.nlargest(1, "char_length")

    print(f"\nShortest example for label={label_value}:")
    print(shortest[["char_length", "word_count", "derived_text"]].to_string(index=False))

    print(f"\nLongest example for label={label_value}:")
    print(longest[["char_length", "word_count", "derived_text"]].to_string(index=False))



Random samples for label=TRUE (n=5):
----------------------------------------------------------------------------------------------------
index: 2253
char_length: 204, word_count: 35, approx_turn_count: 1
Stay abreasted featuring articles on emerging designs like athleading wear's rise or the re-born 80\multum fashion trend, as well as classic fashion staples and timeless pieces that never go out of style.
----------------------------------------------------------------------------------------------------
index: 2124
char_length: 148, word_count: 25, approx_turn_count: 2
Home and kitchen items

"Welcome writer. Whether someone lives or doesn&aposfit, haven’t got you more relaxed and comfortable than you’ve ever been.
----------------------------------------------------------------------------------------------------
index: 393
char_length: 188, word_count: 31, approx_turn_count: 2
A running support shop with knowledgeable staff and a vast name-brand recall should become your runner's 

In [10]:
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())
print("Solution format columns:", solution_df.columns.tolist())

print("\nDoes test have labels?", "label" in test_df.columns)
print("Solution rows:", len(solution_df))
print("Test rows:", len(test_df))
print("Row count match (solution vs test):", len(solution_df) == len(test_df))

print("\nExpected submission columns:", solution_df.columns.tolist())
print("Solution format dtypes:")
print(solution_df.dtypes)


Train columns: ['label', 'text']
Test columns: ['text']
Solution format columns: ['label']

Does test have labels? False
Solution rows: 2100
Test rows: 2100
Row count match (solution vs test): True

Expected submission columns: ['label']
Solution format dtypes:
label    bool
dtype: object


## Manual Sample Review And Updated Hypothesis

Manual inspection of representative train examples indicates that many `TRUE` samples are not always explicit harm in a narrow keyword sense.

Observed pattern (working hypothesis):
- `TRUE` examples often include incoherent, spam-like, suspicious, or semantically broken text behavior.
- `FALSE` examples are usually more coherent and task-like, even when grammar is imperfect.
- This suggests next EDA should explicitly measure text weirdness/noise features and compare them across classes.

Note: this is an observed hypothesis from sample review, not a guaranteed label rule.


In [11]:
def compute_text_weirdness_features(text):
    text = text or ""
    tokens = re.findall(r"\b\w+\b", text.lower())
    token_count = len(tokens)

    if token_count > 0:
        unique_word_ratio = len(set(tokens)) / token_count
        avg_word_length = float(np.mean([len(t) for t in tokens]))
        long_token_ratio = float(np.mean([len(t) >= 12 for t in tokens]))
    else:
        unique_word_ratio = 0.0
        avg_word_length = 0.0
        long_token_ratio = 0.0

    non_alnum_ratio = (sum(1 for ch in text if not ch.isalnum() and not ch.isspace()) / max(len(text), 1))
    digit_symbol_ratio = (sum(1 for ch in text if ch.isdigit() or (not ch.isalnum() and not ch.isspace())) / max(len(text), 1))
    repeated_token_ratio = max(0.0, 1.0 - unique_word_ratio)

    weirdness_proxy = (
        0.35 * repeated_token_ratio
        + 0.25 * non_alnum_ratio
        + 0.20 * digit_symbol_ratio
        + 0.20 * long_token_ratio
    )

    return pd.Series({
        "token_count": token_count,
        "unique_word_ratio": unique_word_ratio,
        "avg_word_length": avg_word_length,
        "long_token_ratio": long_token_ratio,
        "repeated_token_ratio": repeated_token_ratio,
        "non_alnum_ratio": non_alnum_ratio,
        "digit_symbol_ratio": digit_symbol_ratio,
        "weirdness_proxy": weirdness_proxy,
    })


manual_review_df = train_eda[["label_normalized", "derived_text", "char_length", "word_count"]].copy()
manual_review_df = pd.concat(
    [manual_review_df, manual_review_df["derived_text"].apply(compute_text_weirdness_features)],
    axis=1,
)

true_high_weird = (
    manual_review_df[manual_review_df["label_normalized"] == "TRUE"]
    .nlargest(4, "weirdness_proxy")
    .assign(selection_reason="TRUE_high_weirdness")
)

false_low_weird = (
    manual_review_df[manual_review_df["label_normalized"] == "FALSE"]
    .nsmallest(4, "weirdness_proxy")
    .assign(selection_reason="FALSE_low_weirdness")
)

true_random = (
    manual_review_df[manual_review_df["label_normalized"] == "TRUE"]
    .sample(n=2, random_state=SEED)
    .assign(selection_reason="TRUE_random_sample")
)

false_random = (
    manual_review_df[manual_review_df["label_normalized"] == "FALSE"]
    .sample(n=2, random_state=SEED)
    .assign(selection_reason="FALSE_random_sample")
)

manual_examples = pd.concat(
    [true_high_weird, false_low_weird, true_random, false_random],
    axis=0,
).drop_duplicates().reset_index().rename(columns={"index": "train_index"})

manual_examples["text_preview"] = (
    manual_examples["derived_text"]
    .str.replace("\n", " ", regex=False)
    .str.slice(0, 220)
)

display_cols = [
    "train_index",
    "label_normalized",
    "selection_reason",
    "char_length",
    "word_count",
    "unique_word_ratio",
    "repeated_token_ratio",
    "non_alnum_ratio",
    "weirdness_proxy",
    "text_preview",
]

print("Representative manually reviewed examples from train data:")
print(manual_examples[display_cols].to_string(index=False))


Representative manually reviewed examples from train data:
 train_index label_normalized    selection_reason  char_length  word_count  unique_word_ratio  repeated_token_ratio  non_alnum_ratio  weirdness_proxy                                                                                                                                                                                                                  text_preview
        1928             TRUE TRUE_high_weirdness          278         112           0.125000              0.875000         0.025180         0.392401  Thatcher D'souzechua 64470 Aluminé (Argentine Temuco Ar 1 S.O Paulo, SP, Brazil) 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
        2289             TRUE TRUE_high_weirdness          325          36           0.305556              0.694444         0.280000         0.380167 ={}", font_size  def show_regular_groupevent(pr

## EDA Findings

- **Target**: Binary harmful conversation detection with labels in train (`TRUE` = harmful, `FALSE` = non-harmful).
- **Input structure**: Train and test are JSONL with a primary `text` field. In this dataset, `text` values are plain strings (not nested objects/lists), but helper extraction logic should still stay robust for nested variants.
- **Class balance**: Train has 3500 `FALSE` (71.43%) and 1400 `TRUE` (28.57%), so the dataset is moderately imbalanced toward `FALSE`.
- **Modeling text field**: The `text` column is the core feature field for later modeling. A derived text field is useful during EDA only, especially for consistent metric calculations.
- **Risks / patterns noticed**: There are noisy and unusual phrasing patterns, occasional multiline formatting, and mixed writing quality that can hide intent. Label leakage from explicit keywords alone is unlikely to be sufficient.
- **Train/test/submission checks**:
  - Train columns: label, text
  - Test columns: text
  - Test contains label column: no
  - `solution_format.csv` rows = 2100, test rows = 2100 (match: yes)
  - Submission columns expected: label
- **Recommended next step**: Build a reproducible baseline text-classification pipeline using the train `text` field, stratified validation, and careful handling of class imbalance.


In [ ]:
summary_md = """# EDA Summary

## EDA Findings

- **Target**: Binary harmful conversation detection with labels in train (`TRUE` = harmful, `FALSE` = non-harmful).
- **Input structure**: Train and test are JSONL with a primary `text` field. In this dataset, `text` values are plain strings (not nested objects/lists), but helper extraction logic should still stay robust for nested variants.
- **Class balance**: Train has 3500 `FALSE` (71.43%) and 1400 `TRUE` (28.57%), so the dataset is moderately imbalanced toward `FALSE`.
- **Modeling text field**: The `text` column is the core feature field for later modeling. A derived text field is useful during EDA only, especially for consistent metric calculations.
- **Risks / patterns noticed**: There are noisy and unusual phrasing patterns, occasional multiline formatting, and mixed writing quality that can hide intent. Label leakage from explicit keywords alone is unlikely to be sufficient.
- **Train/test/submission checks**:
  - Train columns: label, text
  - Test columns: text
  - Test contains label column: no
  - `solution_format.csv` rows = 2100, test rows = 2100 (match: yes)
  - Submission columns expected: label
- **Recommended next step**: Build a reproducible baseline text-classification pipeline using the train `text` field, stratified validation, and careful handling of class imbalance.

## Manual Sample Review Addendum

- The initial EDA confirmed the core dataset structure, class imbalance, and submission schema expectations.
- Manual reading of train samples suggests `TRUE` is not always direct harmful intent in a narrow toxicity sense.
- Many `TRUE` samples also look abnormal in language behavior: incoherent word mixing, spam-like phrasing, semantic breaks, strange formatting, or suspicious tone.
- `FALSE` samples are often more coherent and purpose-driven, even when grammar quality is imperfect.
- This shifts the near-term analysis direction: treat the problem as harmful/unsafe/abnormal detection, not keyword-level harm detection only.
- Future feature engineering for EDA should measure fluency, coherence, randomness, formatting noise, and suspicious phrasing patterns in addition to explicit harmful cues.
- This is a working hypothesis from observed samples, not a guaranteed labeling rule, and should be validated with broader quantitative analysis.
"""

summary_path = OUTPUT_DIR / "eda_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")
print("Saved:", summary_path)
print("\n" + summary_md)
